In [133]:
from pathlib import Path

import pandas as pd
import numpy as np

In [134]:
### Directories
project_root = Path.cwd()

ground_truth_directory = project_root / Path("eval/gt")
predicted_directory = project_root / Path("eval/pred")
result_directory = project_root / Path("eval/result")

ground_truth_directory.mkdir(parents=True, exist_ok=True)
predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [135]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [136]:
### Main Function
def evaluate_result(truth_csv_path, predicted_csv_path):
    
    ### Initialize Dataframes
    df_truth = pd.read_csv(truth_csv_path)
    df_pred = pd.read_csv(predicted_csv_path)


    ### Clean Up Dataframes
    df_truth.columns = df_truth.columns.str.strip()
    df_pred.columns = df_pred.columns.str.strip()


    ### Merge Both Dataframes
    merged = pd.merge(df_truth, df_pred, on=['frame_index', 'human_id', 'object_id'], how='outer', indicator=True)


    ### Checks for True Positives, False Positives, and False Negatives From The Merge DataFrame 
    tp = (merged['_merge'] == 'both').sum()
    fp = (merged['_merge'] == 'right_only').sum()
    fn = (merged['_merge'] == 'left_only').sum()
    # print(f"True Positives: {tp}")
    # print(f"False Positives: {tp}")
    # print(f"True Negatives: {tp}")


    ### Calculate Precision, Recall, and F1
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * ((precision * recall) / (precision + recall))

    return precision, recall, f1


In [137]:
# video_name = "vid17"

# gt_csv_file = ground_truth_directory / (video_name + "_true_frames.csv")
# pred_csv_file = predicted_directory / (video_name + "_pred_frames.csv")

# precision, recall, f1 = evaluate_result(gt_csv_file, pred_csv_file)

# result_text = f"{video_name:<6} Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}"
# print(result_text)

# with open(result_directory / "result.txt", "w") as result_log:
#     result_log.write(result_text)

In [138]:
results = []

for gt_csv_file in ground_truth_directory.glob("*_true_frames.csv"):

    video_name = gt_csv_file.stem[:-12]
    pred_csv_file = predicted_directory / (video_name + "_pred_frames.csv")

    precision, recall, f1 = evaluate_result(gt_csv_file, pred_csv_file)
    
    result_text = f"{video_name:<6} Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}"
    results.append(result_text)
    print(result_text)

with open(result_directory / "result.txt", "w") as result_log:
    result_log.write("\n".join(results))

vid02  Precision: 1.000 | Recall: 1.000 | F1: 1.000


In [139]:
# ### Main Function
# def evaluate_tiou(truth_csv_path, predicted_csv_path):
    
#     ### Initialize Dataframes
#     df_truth = pd.read_csv(truth_csv_path)
#     df_pred = pd.read_csv(predicted_csv_path)


#     ### Merge Both Dataframes
#     df_merged = pd.merge(
#         df_truth,
#         df_pred,
#         on=['human_id', 'object_id'],
#         how='outer',
#         indicator=True,
#         suffixes=('_gt', '_pred')
#     )
    

#     ### Calculate Intersection Of Frames
#     inter_s = np.maximum(df_merged['frame_start_gt'], df_merged['frame_start_pred'])
#     inter_e = np.minimum(df_merged['frame_end_gt'], df_merged['frame_end_pred'])
#     inter = np.maximum(0, inter_e - inter_s + 1)


#     ### Calculate Union Of Frames
#     union_s = np.minimum(df_merged['frame_start_gt'], df_merged['frame_start_pred'])
#     union_e = np.maximum(df_merged['frame_end_gt'], df_merged['frame_end_pred'])
#     union = union_e - union_s + 1


#     ### Calculate And Return TIoU
#     df_merged['tiou'] = inter / union
#     return df_merged

In [140]:
### Main Function Test
def evaluate_tiou(truth_csv_path, predicted_csv_path):
    

    ### Initialize Dataframes
    df_truth = pd.read_csv(truth_csv_path)
    df_pred = pd.read_csv(predicted_csv_path)


    ### Merge Both Dataframes
    df_merged = pd.merge(
        df_truth,
        df_pred,
        on=['human_id', 'object_id'],
        how='outer',
        indicator=True,
        suffixes=('_gt', '_pred')
    )
    

    ### Aggregate Fragmented Predictions
    mask = (df_merged['frame_start_pred'] <= df_merged['frame_end_gt']) & (df_merged['frame_end_pred'] >= df_merged['frame_start_gt'])
    df_matched = df_merged[mask].groupby(['human_id', 'object_id', 'frame_start_gt', 'frame_end_gt'], as_index=False).agg({'frame_start_pred': 'min', 'frame_end_pred': 'max'})
    df_merged = pd.concat([df_matched, df_merged[~mask]], ignore_index=True)


    ### Calculate Intersection Of Frames
    inter_s = np.maximum(df_merged['frame_start_gt'], df_merged['frame_start_pred'])
    inter_e = np.minimum(df_merged['frame_end_gt'], df_merged['frame_end_pred'])
    inter = np.maximum(0, inter_e - inter_s + 1)


    ### Calculate Union Of Frames
    union_s = np.minimum(df_merged['frame_start_gt'], df_merged['frame_start_pred'])
    union_e = np.maximum(df_merged['frame_end_gt'], df_merged['frame_end_pred'])
    union = union_e - union_s + 1


    ### Calculate And Return TIoU
    df_merged['tiou'] = inter / union
    return df_merged

In [141]:
def convert_to_readable(df_tiou):
    df = df_tiou

    df["frames_gt"] = df["frame_start_gt"].astype(str) + " -> " + df["frame_end_gt"].astype(str)
    df["frames_pred"] = df["frame_start_pred"].astype(str) + " -> " + df["frame_end_pred"].astype(str)

    df_success = df[[
        "human_id",
        "object_id",
        "frames_gt",
        "frames_pred",
        "tiou"
    ]].rename(columns={
        "human_id": "Human ID",
        "object_id": "Object ID",
        "frames_gt": "Frames Truth",
        "frames_pred": "Frames Predicted",
        "tiou": "TIOU"
    })

    return df_success

In [142]:
video_name = "test2"

gt_csv_file = ground_truth_directory / (video_name + "_true_summary.csv")
pred_csv_file = predicted_directory / (video_name + "_pred_summary.csv")

df_tiou = evaluate_tiou(gt_csv_file, pred_csv_file)

df_success = convert_to_readable(df_tiou)

print(df_success)

with open(result_directory / (video_name + "_tiou.txt"), "w") as tiou_log:
    header_text = f"{"Human ID":>10} {"Object ID":>10} {"Frames Truth":>20} {"Frames Predicted":>20} {"TIOU":>12}\n"
    tiou_log.write(header_text)
    tiou_log.writelines([
        f"{human_id:>10} {object_id:>10} {frames_truth:>20} {frames_pred:>20} {tiou:>12.5f}\n"
        for _, (human_id, object_id, frames_truth, frames_pred, tiou) in df_success.iterrows()
    ])



# for index, row in df_success.iterrows():
#     print(row.values)

   Human ID  Object ID Frames Truth Frames Predicted      TIOU
0         1          2      0 -> 20          0 -> 20  1.000000
1         3          4     0 -> 100         0 -> 100  1.000000
2         5          6      0 -> 50         10 -> 60  0.672131
